In [21]:
print("Agentic RAG with LlamaIdex")

Agentic RAG with LlamaIdex


In [22]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")

In [23]:
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model = 'models/gemini-2.0-flash'
)

In [ ]:
response = llm.complete("tell me about agentic rag")
print(response)

"Agentic rag" isn't a widely recognized or established term in the fields of artificial intelligence, natural language processing, or information retrieval. It's possible it's a newly coined term, a niche concept, or a misspelling.

However, we can break down the potential meaning based on the individual words:

*   **Agentic:** This refers to the property of being an "agent," meaning something that acts or exerts power. In AI, an agent is often a system that can perceive its environment, make decisions, and take actions to achieve goals. An agentic system is proactive, autonomous, and goal-oriented.

*   **RAG:** This stands for **Retrieval-Augmented Generation**. It's a technique in natural language processing where a language model (like a large language model or LLM) retrieves information from an external knowledge source (like a database, a collection of documents, or the internet) and uses that information to generate more accurate, relevant, and informative text.

Therefore, "ag

In [25]:
# Load Data
from llama_index.core import SimpleDirectoryReader

document = SimpleDirectoryReader(input_files=["../Data/CAG.pdf"]).load_data()

In [ ]:
# Use it for splitting the sentence
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(document)

In [29]:
from llama_index.embeddings.gemini import GeminiEmbedding

embed_model = GeminiEmbedding(
    model = 'models/embedding-001',
)

In [ ]:
# Define Summary Index adnd Vectore Index over the Same data
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes, embed_model=embed_model) # add embed_model explicitly as default it uses OpenAI

In [ ]:
# Define Query Engine and Set Metadata
# add llm explicitly as default it uses OpenAI
summary_query_engine = summary_index.as_query_engine(
    response_mode='tree_summarize',
    up_sync= True, # faster query in async mode
    llm=llm
)

vector_query_engine = vector_index.as_query_engine(llm=llm)

In [35]:
from llama_index.core.tools import QueryEngineTool

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description="Useful for summarization questions related to CAG",
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description="Useful for Retrieving specific conext from CAG"
)

In [41]:
# Router Query Engine
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# add llm = llm explicitly if not using OpenAI
query_engine = RouterQueryEngine(
    selector= LLMSingleSelector.from_defaults(llm=llm),
    llm=llm,
    query_engine_tools = [
        summary_tool,
        vector_tool,
    ],
    verbose=True
)


In [45]:
import pprint as pp

response = query_engine.query("What is the summary of the document?")
pp.pprint(response)

Selecting query engine 0: The question 'What is the summary of the document?' directly asks for a summarization, making choice 1, which is 'Useful for summarization questions related to CAG', the most relevant..
Response(response='This paper introduces cache-augmented generation (CAG) as '
                  'an alternative to retrieval-augmented generation (RAG) for '
                  'knowledge tasks, especially when dealing with a limited '
                  'knowledge base. CAG involves preloading relevant resources '
                  "into the LLM's extended context and caching its runtime "
                  'parameters, eliminating real-time retrieval and its '
                  'associated challenges like latency and retrieval errors. '
                  'The study includes comparative analyses that show '
                  'long-context LLMs can outperform or complement traditional '
                  'RAG pipelines in certain scenarios. Experiments conducted '
              

In [47]:
response = query_engine.query("How to implemt CAG from the document?")
pp.pprint(response)

Selecting query engine 0: The question 'How to implement CAG from the document?' is essentially asking for a summary of the implementation process. Option 1 is described as 'Useful for summarization questions related to CAG', making it the most relevant choice..
Response(response='The Cache-Augmented Generation (CAG) framework involves '
                  'three phases:\n'
                  '\n'
                  '1.  **External Knowledge Preloading**: Relevant documents '
                  'are preprocessed and formatted to fit within the language '
                  "model's context window. The language model then processes "
                  'these documents, transforming them into a precomputed '
                  'key-value (KV) cache. This KV cache is stored for future '
                  'use, and the computational cost of processing the documents '
                  'is incurred only once.\n'
                  '2.  **Inference**: During inference, the precomputed KV '
        

In [50]:
response = query_engine.query("Give me list of reference from above document")
pp.pprint(response)

Selecting query engine 1: The question 'Give me list of reference from above document' requires retrieving specific context (references) from the document, which aligns with the description of choice 2..
Response(response='*   [8] Tianyi Zhang, Varsha Kishore, Felix Wu, Kilian Q '
                  'Weinber ger, and Yoav Artzi.\n'
                  '*   BERTScore: Evaluating Text Generation with BERT. I n '
                  'International Conference on Learning Representations.\n',
         source_nodes=[NodeWithScore(node=TextNode(id_='b2b0530a-d191-48b4-9116-781bb2db3707', embedding=None, metadata={'page_label': '5', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_si